# BACE general BFS W scan — Q=16, T=16, W=0.5

This notebook is the local executable code for one minimum W-scan folder.
Only `W` differs across sibling folders; tokens, splits, `g`, `L`, QRC/CRC
readout, raw baselines, and the single linear readout are protocol-identical.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import torch

SETTING_DIR = Path(r"/home/ubuntu/Workspace/Fei/wlf/Article/QRC/FINAL/2_linear_readout/general/BACE/03_W_scan/Q16_T16/W0p5").resolve()
PLAN_PATH = SETTING_DIR / "setting_plan.json"
PROJECT_ROOT = Path(r"/home/ubuntu/Workspace/Fei/wlf/Article/QRC/FINAL/2_linear_readout").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from core.linear_readout import run_setting_plan, plot_single_setting_summary

print("setting:", SETTING_DIR)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())


In [ ]:
RUN_QRC = True
FORCE = False

plan = json.loads(PLAN_PATH.read_text(encoding="utf-8"))
display(pd.Series(plan, name="setting_plan"))


## Protocol locked in this folder

- Dataset/token stream: BACE, general task-independent local-BFS events.
- Token slice: first `16` events and first `48` token coordinates.
- QRC/CRC scale: `W=0.5`, `g=0.84`, `L=3`.
- QRC readout: Z + all-ZZ; CRC3 matched readout: direct z + all zz from a Q×3 state (no PCA); CRC readout: scalar-Q state + all pair products.
- Baselines: raw-full flattened tokens and train-split raw-PCA.
- Classifier: exactly one `torch.nn.Linear(d, 1)`, no standardization, no hidden layer, no regularization.
- Seeds/splits: `0,10,...,90`; QRC/CRC parameters, scaffold split and training all change with seed.
- Best epoch: validation `AUC > loss > ACC`.


In [ ]:
metrics, summary = run_setting_plan(
    PLAN_PATH,
    run_qrc=RUN_QRC,
    force=FORCE,
)

display(summary)
display(metrics.sort_values(["method", "seed"]).tail(12))


In [ ]:
fig = plot_single_setting_summary(SETTING_DIR, plan["dataset"])
plt.show()

print("summary:", SETTING_DIR / "performance_summary.csv")
print("predictions:", SETTING_DIR / "test_predictions.csv")
